In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path(
    r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump"
)

INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "integrated" / "crop_soil_climate_complete_2013_2025.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "integrated" / "crop_soil_climate_complete_corrected_2013_2025.csv"
AUDIT_FILE = PROJECT_ROOT / "data" / "processed" / "integrated" / "soil_imputation_correction_audit.csv"

# Fallback for this notebook when run from the provided execution environment.
if not INPUT_FILE.exists():
    fallback_input = Path("/mnt/data/crop_soil_climate_complete_2013_2025.csv")
    if fallback_input.exists():
        INPUT_FILE = fallback_input
        OUTPUT_FILE = Path("/mnt/data/crop_soil_climate_complete_corrected_2013_2025.csv")
        AUDIT_FILE = Path("/mnt/data/soil_imputation_correction_audit.csv")

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)
print("Audit :", AUDIT_FILE)


Input : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_2013_2025.csv
Output: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_corrected_2013_2025.csv
Audit : C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\soil_imputation_correction_audit.csv


In [2]:
df = pd.read_csv(INPUT_FILE)

assert len(df) == 67826, f"Expected 67,826 rows, found {len(df)}"

soil_fraction_cols = [
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction",
]

required_cols = soil_fraction_cols + [
    "soil_type",
    "state",
    "district",
    "soil_data_status",
]

missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

print("Dataset shape:", df.shape)


Dataset shape: (67826, 52)


In [3]:
# Normalize state only for grouping/reference lookup.
df["_state_key_correction"] = (
    df["state"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

imputed_soil_mask = df["soil_data_status"].eq("imputed")

for col in soil_fraction_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Observed/recovered soil rows:", int((~imputed_soil_mask).sum()))
print("Imputed soil rows           :", int(imputed_soil_mask.sum()))


Observed/recovered soil rows: 55889
Imputed soil rows           : 11937


In [4]:

#Diagnose source texture sums
df["_texture_sum_before"] = df[soil_fraction_cols].sum(axis=1)

print("Observed/recovered texture-sum statistics:")
display(df.loc[~imputed_soil_mask, "_texture_sum_before"].describe())

print("\nImputed texture-sum statistics before correction:")
display(df.loc[imputed_soil_mask, "_texture_sum_before"].describe())

print("\nObserved/recovered rows with texture sum < 1:",
      int((df.loc[~imputed_soil_mask, "_texture_sum_before"] < 1 - 1e-6).sum()))

print("Observed/recovered rows with texture sum approximately 1:",
      int(np.isclose(
          df.loc[~imputed_soil_mask, "_texture_sum_before"], 1.0, atol=1e-6
      ).sum()))


Observed/recovered texture-sum statistics:


count    55889.000000
mean         0.810104
std          0.147353
min          0.000000
25%          0.753973
50%          0.847922
75%          0.913121
max          0.994699
Name: _texture_sum_before, dtype: float64


Imputed texture-sum statistics before correction:


count    11937.000000
mean         0.764250
std          0.124603
min          0.074880
25%          0.731450
50%          0.797488
75%          0.819763
max          0.912232
Name: _texture_sum_before, dtype: float64


Observed/recovered rows with texture sum < 1: 55889
Observed/recovered rows with texture sum approximately 1: 0


In [5]:
#Build reference compositions
reference = df.loc[
    ~imputed_soil_mask,
    ["_state_key_correction"] + soil_fraction_cols
].copy()

reference[soil_fraction_cols] = reference[soil_fraction_cols].clip(0, 1)

state_medians = (
    reference
    .dropna(subset=["_state_key_correction"])
    .groupby("_state_key_correction")[soil_fraction_cols]
    .median()
)

national_median = reference[soil_fraction_cols].median()

print("States with observed/recovered soil reference:", len(state_medians))
print("\nNational median vector:")
display(national_median)


States with observed/recovered soil reference: 34

National median vector:


clayey_fraction             0.279456
clayey_skeletal_fraction    0.000000
loamy_fraction              0.437548
sandy_fraction              0.000000
dtype: float64

In [6]:
def normalize_texture_vector(values):
    values = pd.Series(values, index=soil_fraction_cols, dtype="float64").clip(lower=0)
    total = values.sum()

    if not np.isfinite(total) or total <= 0:
        raise ValueError("Cannot normalize a soil texture vector with non-positive/invalid sum.")

    return values / total


national_profile = normalize_texture_vector(national_median)
state_profiles = state_medians.apply(normalize_texture_vector, axis=1)

print("Normalized national profile:")
display(national_profile)

print("\nReference profile sum check:")
print("National sum:", national_profile.sum())
print("State sums min:", state_profiles.sum(axis=1).min())
print("State sums max:", state_profiles.sum(axis=1).max())

assert np.isclose(national_profile.sum(), 1.0, atol=1e-10)
assert np.allclose(state_profiles.sum(axis=1), 1.0, atol=1e-10)


Normalized national profile:


clayey_fraction             0.389755
clayey_skeletal_fraction    0.000000
loamy_fraction              0.610245
sandy_fraction              0.000000
dtype: float64


Reference profile sum check:
National sum: 1.0
State sums min: 0.9999999999999998
State sums max: 1.0


In [7]:
# Preserve observed/recovered values for an exact before/after comparison.
observed_before = df.loc[~imputed_soil_mask, soil_fraction_cols].copy()

df["soil_imputation_method"] = "not_imputed"

for idx in df.index[imputed_soil_mask]:
    state_key = df.at[idx, "_state_key_correction"]

    if pd.notna(state_key) and state_key in state_profiles.index:
        profile = state_profiles.loc[state_key]
        method = "state_reference_normalized"
    else:
        profile = national_profile
        method = "national_reference_normalized"

    df.loc[idx, soil_fraction_cols] = profile.values
    df.at[idx, "soil_imputation_method"] = method

print(df["soil_imputation_method"].value_counts(dropna=False))


soil_imputation_method
not_imputed                      55889
state_reference_normalized       11685
national_reference_normalized      252
Name: count, dtype: int64


In [8]:

# Correct soil_type only where soil data was imputed
texture_to_soil_type = {
    "clayey_fraction": "Clayey",
    "clayey_skeletal_fraction": "Clayey skeletal",
    "loamy_fraction": "Loamy",
    "sandy_fraction": "Sandy",
}

def dominant_soil_type(row):
    return texture_to_soil_type[
        row[soil_fraction_cols].astype(float).idxmax()
    ]

df.loc[imputed_soil_mask, "soil_type"] = (
    df.loc[imputed_soil_mask]
      .apply(dominant_soil_type, axis=1)
)

print("Imputed soil_type after correction:")
print(df.loc[imputed_soil_mask, "soil_type"].value_counts(dropna=False))


Imputed soil_type after correction:
soil_type
Loamy     6765
Clayey    5172
Name: count, dtype: int64


In [9]:
# Validate all soil fractions are physically bounded.
df[soil_fraction_cols] = df[soil_fraction_cols].clip(0, 1)

assert not df[soil_fraction_cols].isna().any().any(),     "Missing soil fractions remain."

for col in soil_fraction_cols:
    assert (df[col] >= 0).all() and (df[col] <= 1).all(),         f"Invalid values in {col}"

texture_sum_after = df[soil_fraction_cols].sum(axis=1)

imputed_sum_after = texture_sum_after.loc[imputed_soil_mask]

print("Imputed texture-sum statistics AFTER correction:")
display(imputed_sum_after.describe())

print("\nImputed texture sum min:", imputed_sum_after.min())
print("Imputed texture sum max:", imputed_sum_after.max())

assert np.allclose(imputed_sum_after, 1.0, atol=1e-6),     "Some imputed soil compositions do not sum to 1."

print("\nPASS: all imputed soil compositions sum to 1.")


Imputed texture-sum statistics AFTER correction:


count    1.193700e+04
mean     1.000000e+00
std      8.578349e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64


Imputed texture sum min: 0.9999999999999998
Imputed texture sum max: 1.0

PASS: all imputed soil compositions sum to 1.


In [10]:
# Verify observed/recovered soil fractions were not changed.
observed_after = df.loc[~imputed_soil_mask, soil_fraction_cols].copy()

comparison = (
    observed_before.reset_index(drop=True)
    .compare(observed_after.reset_index(drop=True))
)

print("Changed observed/recovered soil fraction cells:", len(comparison))

assert comparison.empty,     "Observed/recovered soil fractions were changed."

print("PASS: observed/recovered soil fractions are unchanged.")


Changed observed/recovered soil fraction cells: 0
PASS: observed/recovered soil fractions are unchanged.


In [11]:
valid_soil_types = {
    "Clayey",
    "Clayey skeletal",
    "Loamy",
    "Sandy",
}

assert df["soil_type"].notna().all(), "Missing soil_type remains."

invalid_soil_types = set(df["soil_type"].dropna().unique()) - valid_soil_types

assert not invalid_soil_types,     f"Unexpected soil_type values: {invalid_soil_types}"

assert len(df) == 67826

print("PASS: soil types and row count are valid.")


PASS: soil types and row count are valid.


In [12]:

# Final audit
audit = pd.DataFrame([
    {
        "metric": "total_rows",
        "value": len(df),
    },
    {
        "metric": "observed_or_recovered_soil_rows",
        "value": int((~imputed_soil_mask).sum()),
    },
    {
        "metric": "imputed_soil_rows",
        "value": int(imputed_soil_mask.sum()),
    },
    {
        "metric": "imputed_texture_sum_min_after",
        "value": float(imputed_sum_after.min()),
    },
    {
        "metric": "imputed_texture_sum_max_after",
        "value": float(imputed_sum_after.max()),
    },
    {
        "metric": "imputed_rows_not_summing_to_1_after",
        "value": int((~np.isclose(imputed_sum_after, 1.0, atol=1e-6)).sum()),
    },
    {
        "metric": "changed_observed_fraction_cells",
        "value": int(len(comparison)),
    },
])

for method, count in df["soil_imputation_method"].value_counts(dropna=False).items():
    audit.loc[len(audit)] = {
        "metric": f"soil_imputation_method::{method}",
        "value": int(count),
    }

display(audit)


,metric,value
0,total_rows,67826.0
1,observed_or_recovered_soil_rows,55889.0
2,imputed_soil_rows,11937.0
3,imputed_texture_sum_min_after,1.0
4,imputed_texture_sum_max_after,1.0
5,imputed_rows_not_summing_to_1_after,0.0
6,changed_observed_fraction_cells,0.0
7,soil_imputation_method::not_imputed,55889.0
8,soil_imputation_method::state_reference_normal...,11685.0
9,soil_imputation_method::national_reference_nor...,252.0


In [13]:
# Remove temporary diagnostic columns.
df.drop(columns=["_state_key_correction", "_texture_sum_before"], inplace=True)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_FILE, index=False)
audit.to_csv(AUDIT_FILE, index=False)

print("Corrected dataset saved to:")
print(OUTPUT_FILE)

print("\nAudit saved to:")
print(AUDIT_FILE)

print("\nCorrected dataset shape:", df.shape)
print("File size (MB):", round(OUTPUT_FILE.stat().st_size / (1024**2), 2))


Corrected dataset saved to:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_corrected_2013_2025.csv

Audit saved to:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\soil_imputation_correction_audit.csv

Corrected dataset shape: (67826, 53)
File size (MB): 26.68
